# Narrative Timeline: Perkembangan Vibe Coding

Tahap 1: **Volume Trend per Periode** (bulanan/kuartalan) + deteksi spike sebagai kandidat event.

Jalankan cell secara berurutan dari atas.

In [ ]:
import os
import io
import re
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "data/raw/vibecoding_relevant_10000.csv"
df = None

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("File terbaca langsung dari runtime ini (kernel lokal).")
    print("Jumlah baris:", len(df))
else:
    print("File tidak ada di runtime ini (kemungkinan kernel Colab remote).")
    print("Klik tombol Upload di bawah, pilih CSV, lalu jalankan CELL BERIKUTNYA.")
    import ipywidgets as widgets
    from IPython.display import display

    uploader = widgets.FileUpload(accept=".csv", multiple=False)
    display(uploader)

In [ ]:
if df is None:
    if not uploader.value:
        raise RuntimeError("Belum ada file diupload. Klik Upload di cell sebelumnya, lalu jalankan cell ini lagi.")

    uploaded_files = uploader.value
    if isinstance(uploaded_files, dict):
        filename, file_info = next(iter(uploaded_files.items()))
        content = file_info["content"]
    else:
        file_info = uploaded_files[0]
        filename = file_info["name"]
        content = file_info["content"]

    df = pd.read_csv(io.BytesIO(bytes(content)))
    print("Berhasil membaca:", filename)
    print("Jumlah baris:", len(df))

df.head()

## 1. Volume Trend per Periode (Bulanan & Kuartalan)

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
df = df.dropna(subset=["created_at"]).copy()

df["month"] = df["created_at"].dt.to_period("M")
df["quarter"] = df["created_at"].dt.to_period("Q")

monthly = df.groupby("month").size()
quarterly = df.groupby("quarter").size()

print("Rentang data:", df["created_at"].min().date(), "->", df["created_at"].max().date())
print("Jumlah bulan dengan data:", len(monthly))
print()
print("Volume per kuartal:")
print(quarterly)

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

monthly.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Volume Tweet Vibe Coding per Bulan")
axes[0].set_xlabel("")
axes[0].set_ylabel("Jumlah tweet")
axes[0].tick_params(axis="x", rotation=90, labelsize=8)

quarterly.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Volume Tweet Vibe Coding per Kuartal")
axes[1].set_xlabel("Periode")
axes[1].set_ylabel("Jumlah tweet")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 2. Pertumbuhan & Kurva Adopsi Kumulatif

In [ ]:
monthly_df = monthly.to_frame("jumlah")
monthly_df["pct_change"] = monthly_df["jumlah"].pct_change() * 100
monthly_df["kumulatif"] = monthly_df["jumlah"].cumsum()
monthly_df["pct_dari_total"] = monthly_df["jumlah"] / monthly_df["jumlah"].sum() * 100

print("Volume bulanan + pertumbuhan:")
print(monthly_df.round(1))

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(range(len(monthly)), monthly.values, marker="o", color="steelblue")
axes[0].set_title("Tren Volume Bulanan")
axes[0].set_ylabel("Jumlah tweet")
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels([str(m) for m in monthly.index], rotation=90, fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(range(len(monthly_df)), monthly_df["kumulatif"].values, marker="o", color="green")
axes[1].set_title("Akumulasi Kumulatif Tweet (bentuk kurva adopsi)")
axes[1].set_ylabel("Kumulatif tweet")
axes[1].set_xticks(range(len(monthly)))
axes[1].set_xticklabels([str(m) for m in monthly.index], rotation=90, fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Deteksi Spike Harian (Kandidat Event)

Spike dihitung dengan membandingkan volume harian terhadap baseline rolling median 14 hari (z-score).

In [ ]:
daily = df.groupby(df["created_at"].dt.date).size()
daily.index = pd.to_datetime(daily.index)
daily = daily.asfreq("D", fill_value=0)

rolling_median = daily.rolling(14, center=True, min_periods=3).median()
rolling_std = daily.rolling(14, center=True, min_periods=3).std().replace(0, 1)
z_score = (daily - rolling_median) / rolling_std

SPIKE_Z = 2.0
spikes = daily[(z_score > SPIKE_Z) & (daily > daily.median() * 1.5)]

print(f"Jumlah hari terdeteksi spike (z > {SPIKE_Z}): {len(spikes)}")
print()
print("=== TOP 15 SPIKE (kandidat event penting) ===")
top_spikes = spikes.sort_values(ascending=False).head(15).sort_index()
for tanggal, jumlah in top_spikes.items():
    print(f"{tanggal.date()}  ->  {jumlah} tweet  (z={z_score[tanggal]:.1f})")

plt.figure(figsize=(14, 5))
plt.plot(daily.index, daily.values, linewidth=0.9, label="Tweet per hari", color="steelblue")
plt.plot(rolling_median.index, rolling_median.values, linewidth=1.5, linestyle="--",
         label="Baseline (median 14 hari)", color="gray")
plt.scatter(top_spikes.index, top_spikes.values, color="red", zorder=5, s=45, label="Spike terdeteksi")
for tanggal, jumlah in top_spikes.items():
    plt.annotate(tanggal.strftime("%Y-%m-%d"), (tanggal, jumlah), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=7, rotation=45)
plt.title("Timeline Harian Vibe Coding + Spike Terdeteksi")
plt.xlabel("Tanggal")
plt.ylabel("Jumlah tweet")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Tweet Representatif per Spike

Untuk mengidentifikasi event apa yang menyebabkan lonjakan pada tanggal tersebut.

In [ ]:
for tanggal in top_spikes.index:
    hari_itu = df[df["created_at"].dt.date == tanggal.date()]
    print()
    print("=" * 70)
    print(f"{tanggal.date()}  ({len(hari_itu)} tweet)")
    print("=" * 70)
    for text in hari_itu["text"].head(5):
        print("-", str(text)[:180])